In [7]:

import os
import sys
import subprocess

print("[Setup] Running system packaging alignments for pytorch-crf...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "pytorch-crf", "--user", "--quiet"])

import site
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

os.environ["HF_TOKEN"] = "HUGGING FACE TOKEN" 

import re
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

try:
    from datasets import load_dataset
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "--user", "--quiet"])
    from datasets import load_dataset

CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 1,
    "chunk_size": 4,         
    "epochs": 15,            
    "tokenizer": AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
}

label2id = {
    'ANALYSIS': 0, 'ARG_PETITIONER': 1, 'ARG_RESPONDENT': 2, 'FAC': 3, 
    'ISSUE': 4, 'NONE': 5, 'PREAMBLE': 6, 'PRE_NOT_RELIED': 7, \
    'PRE_RELIED': 8, 'RATIO': 9, 'RLC': 10, 'RPC': 11, 'STA': 12, 'None': 13
}
id2label = {v: k for k, v in label2id.items()}
CONFIG["num_labels"] = len(label2id)

print(f"\n[Success] Environment aligned! Targeting Device: {CONFIG['device']}")

[Setup] Running system packaging alignments for pytorch-crf...

[Success] Environment aligned! Targeting Device: cuda


In [8]:
class LegalSegDataset(Dataset):
    def __init__(self, documents):
        self.documents = documents

    def __len__(self):
        return len(self.documents)

    def __getitem__(self, idx):
        return self.documents[idx]

def identity_collate(batch):
    """ Bypasses default collation to keep document sequence boundaries intact """
    return batch[0]

print("Dataset utility handlers compiled successfully.")

Dataset utility handlers compiled successfully.


In [9]:

def load_opennyai_data(split_name="train"):
    print(f"[Data] Fetching opennyaiorg/InRhetoricalRoles ({split_name} split)...")
    hf_split = "dev" if split_name == "val" else split_name
    hf_dataset = load_dataset("opennyaiorg/InRhetoricalRoles", split=hf_split)
    
    parsed_documents = []
    for idx, row in enumerate(hf_dataset):
        full_text = row["data"]["text"]
        annotations_list = row["annotations"]
        extracted_segments = []
        
        if annotations_list and len(annotations_list) > 0:
            results = annotations_list[0].get("result", [])
            for res in results:
                value = res.get("value", {})
                start_char = value.get("start")
                end_char = value.get("end")
                labels = value.get("labels", ["None"])
                label = labels[0] if labels else "None"
                
                segment_text = full_text[start_char:end_char].strip()
                if segment_text:
                    extracted_segments.append((segment_text, label))
                    
        doc_sentences = []
        doc_labels = []
        
        if extracted_segments:
            for seg_text, label in extracted_segments:
                sub_sentences = re.split(r'(?<=[.!?])\s+', seg_text)
                for sentence in sub_sentences:
                    sentence = sentence.strip()
                    if len(sentence) > 3:
                        doc_sentences.append(sentence)
                        doc_labels.append(label)
        else:
            sub_sentences = re.split(r'(?<=[.!?])\s+', full_text)
            for sentence in sub_sentences:
                sentence = sentence.strip()
                if len(sentence) > 3:
                    doc_sentences.append(sentence)
                    doc_labels.append("None")
                    
        if doc_sentences:
            parsed_documents.append({
                "sentences": doc_sentences,
                "labels": doc_labels,
                "doc_id": f"opennyai_{split_name}_{idx}"
            })
            
    print(f" -> Compiled {len(parsed_documents)} document sequences.")
    return parsed_documents

In [10]:


class NativeCRFLayer(nn.Module):
    def __init__(self, num_tags):
        super(NativeCRFLayer, self).__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))
        
    def _forward_alg(self, emissions):
        seq_len = emissions.size(1)
        init_alphas = torch.full((1, self.num_tags), -10000.0).to(emissions.device)
        init_alphas[0, 0] = 0.0 
        forward_var = init_alphas
        for i in range(seq_len):
            emit_score = emissions[:, i, :].view(-1, self.num_tags, 1)
            next_tag_var = forward_var.view(-1, 1, self.num_tags) + self.transitions + emit_score
            forward_var = torch.logsumexp(next_tag_var, dim=2)
        return torch.logsumexp(forward_var, dim=1)

    def _score_sentence(self, emissions, tags):
        score = torch.zeros(1).to(emissions.device)
        tags = torch.cat([torch.tensor([0], dtype=torch.long).to(tags.device), tags[0]])
        for i, emit in enumerate(emissions[0]):
            score = score + self.transitions[tags[i + 1], tags[i]] + emit[tags[i + 1]]
        return score

    def decode(self, emissions):
        seq_len = emissions.size(1)
        viterbi_vars = torch.full((1, self.num_tags), -10000.0).to(emissions.device)
        viterbi_vars[0, 0] = 0.0
        backpointers = []
        for i in range(seq_len):
            emit_score = emissions[:, i, :].view(-1, self.num_tags, 1)
            next_tag_var = viterbi_vars.view(-1, 1, self.num_tags) + self.transitions + emit_score
            max_vars, bptrs = torch.max(next_tag_var, dim=2)
            viterbi_vars = max_vars
            backpointers.append(bptrs.squeeze(0))
            
        best_score, best_tag_id = torch.max(viterbi_vars, dim=1)
        best_path = [best_tag_id.item()]
        for bptrs_t in reversed(backpointers):
            best_tag_id = bptrs_t[best_path[-1]]
            best_path.append(best_tag_id.item())
        best_path.pop() 
        best_path.reverse()
        return [best_path]

    def forward(self, emissions, tags=None):
        if tags is not None:
            return self._forward_alg(emissions) - self._score_sentence(emissions, tags)
        return self.decode(emissions)


class PaperKillerTransformer(nn.Module):
    def __init__(self, config, label2id):
        super(PaperKillerTransformer, self).__init__()
        self.config = config
        self.label2id = label2id
        self.num_labels = config["num_labels"]
        
        self.alpha = config.get("alpha", 0.7)
        self.beta = config.get("beta", 0.3)
        
        self.legalbert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased")
        self.legalbert.gradient_checkpointing_enable() 
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=768, nhead=12, dim_feedforward=3072, dropout=0.15, batch_first=True
        )
        self.global_context_transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        
        self.feature_dense = nn.Sequential(
            nn.Linear(768, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.1)
        )
        self.classifier = nn.Linear(512, self.num_labels)
        self.crf = NativeCRFLayer(num_tags=self.num_labels)

    def _compute_label_shift_penalty(self, logits, target_ids):
        """ Parallel Track: Penalizes the model dynamically on sequence boundaries """
        seq_len = len(target_ids)
        if seq_len <= 1:
            return torch.tensor(0.0).to(logits.device)
            
        true_shifts = torch.tensor([
            1.0 if target_ids[t] != target_ids[t-1] else 0.0 
            for t in range(1, seq_len)
        ]).to(logits.device)
        
        probs = F.softmax(logits.squeeze(0), dim=-1)
        pred_probs_t = probs[1:]
        pred_probs_t_minus_1 = probs[:-1]
        
        cos_sim = F.cosine_similarity(pred_probs_t, pred_probs_t_minus_1, dim=-1)
        pred_shifts = 1.0 - cos_sim 
        
        return F.mse_loss(pred_shifts, true_shifts)
        
    def forward(self, batch, device, labels=None, class_weights=None):
        sentences = batch["sentences"]
        cls_embeddings = []
        chunk_size = self.config.get("chunk_size", 4)
        
        for i in range(0, len(sentences), chunk_size):
            chunk = sentences[i : i + chunk_size]
            inputs = self.config["tokenizer"](
                chunk, padding=True, truncation=True, max_length=512, return_tensors="pt"
            ).to(device)
            
            outputs = self.legalbert(**inputs)
            cls_vectors = outputs.last_hidden_state[:, 0, :]
            cls_embeddings.append(cls_vectors)
            
        document_tensor = torch.cat(cls_embeddings, dim=0).unsqueeze(0)
        
        enriched_context = self.global_context_transformer(document_tensor)
        features = self.feature_dense(enriched_context)
        logits = self.classifier(features)
        
        predictions = self.crf(logits)
        
        if labels is not None or ("labels" in batch and batch["labels"] is not None):
            target_labels = labels if labels is not None else batch["labels"]
            target_ids = [self.label2id[l] for l in target_labels]
            target_tensor = torch.tensor([target_ids], dtype=torch.long).to(device)
            
            crf_loss = self.crf(logits, target_tensor)
            
            if class_weights is not None:
                penalty_weight = torch.mean(torch.tensor([class_weights[tid] for tid in target_ids]).to(device))
                crf_loss = crf_loss * penalty_weight
                
            shift_loss = self._compute_label_shift_penalty(logits, target_ids)
            
            total_loss = (self.alpha * crf_loss) + (self.beta * shift_loss * 100.0)
            
            return {"loss": total_loss, "crf_loss": crf_loss, "shift_loss": shift_loss, "predictions": predictions}
        else:
            return {"predictions": predictions}

print("[Success] Unfrozen Tokenizer-Aware Parallel Label-Shift Framework Ready.")

[Success] Unfrozen Tokenizer-Aware Parallel Label-Shift Framework Ready.


In [12]:
CONFIG["alpha"] = 0.7
CONFIG["beta"] = 0.3

train_docs = load_opennyai_data(split_name="train")
val_docs = load_opennyai_data(split_name="val")

train_dataset = LegalSegDataset(train_docs)
val_dataset = LegalSegDataset(val_docs)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=identity_collate)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=identity_collate)

class_counts = torch.zeros(CONFIG["num_labels"])
for doc in train_docs:
    for label in doc["labels"]:
        class_counts[label2id[label]] += 1
class_counts = torch.clamp(class_counts, min=1.0)
class_weights = 1.0 / class_counts 
class_weights = (class_weights / class_weights.sum()) * CONFIG["num_labels"]

device = CONFIG["device"]
model = PaperKillerTransformer(CONFIG, label2id)
model.to(device)

optimizer = torch.optim.AdamW([
    {"params": model.legalbert.parameters(), "lr": 2e-6},                  
    {"params": model.global_context_transformer.parameters(), "lr": 2e-5}, 
    {"params": model.feature_dense.parameters(), "lr": 5e-5},               
    {"params": model.classifier.parameters(), "lr": 5e-5},    {"params": model.crf.parameters(), "lr": 1e-3}                         
], weight_decay=0.01)

def train_parallel_pipeline(model, train_loader, val_loader, optimizer, config, weights):
    best_val_loss = float("inf")
    device = config["device"]
    
    print(f"\nLaunching Boundary-Aware Custom Parallel Optimization for {config['epochs']} Epochs...")
    print("=" * 75)
    
    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}", dynamic_ncols=True)
        
        for batch in progress_bar:
            optimizer.zero_grad()
            outputs = model(batch, device, labels=batch.get("labels"), class_weights=weights)
            loss = outputs["loss"]
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            
            progress_bar.set_postfix({
                "Total_L": f"{loss.item():.2f}",
                "CRF_L": f"{outputs['crf_loss'].item():.1f}",
                "Shift_L": f"{outputs['shift_loss'].item():.4f}"
            })
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                outputs = model(batch, device, labels=batch.get("labels"), class_weights=weights)
                val_loss += outputs["loss"].item()
                
        avg_val = val_loss / len(val_loader)
        print(f"-> Epoch {epoch+1} Complete | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {avg_val:.4f}")
        
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), "best_transformer_model.pth")
            print("✨ SOTA Boundary-Aware model updated in 'best_transformer_model.pth'")

train_parallel_pipeline(model, train_loader, val_loader, optimizer, CONFIG, class_weights)

[Data] Fetching opennyaiorg/InRhetoricalRoles (train split)...
 -> Compiled 247 document sequences.
[Data] Fetching opennyaiorg/InRhetoricalRoles (val split)...
 -> Compiled 30 document sequences.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Launching Boundary-Aware Custom Parallel Optimization for 15 Epochs...


Epoch 1/15:   0%|          | 0/247 [00:00<?, ?it/s]

...

-> Epoch 13 Complete | Train Loss: 20.9377 | Val Loss: 13.5775
✨ SOTA Boundary-Aware model updated in 'best_transformer_model.pth'


Epoch 14/15:   0%|          | 0/247 [00:00<?, ?it/s]

-> Epoch 14 Complete | Train Loss: 18.5398 | Val Loss: 13.0402
✨ SOTA Boundary-Aware model updated in 'best_transformer_model.pth'


Epoch 15/15:   0%|          | 0/247 [00:00<?, ?it/s]

-> Epoch 15 Complete | Train Loss: 16.6831 | Val Loss: 12.6942
✨ SOTA Boundary-Aware model updated in 'best_transformer_model.pth'


In [16]:
print("\n" + "="*40)
print(" LAUNCHING FINAL TRANS-MODEL BENCHMARK TEST EVALUATION")
print("="*40)

test_docs = load_opennyai_data(split_name="test")

test_dataset = LegalSegDataset(test_docs)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=identity_collate)

model.eval()

true_labels = []
pred_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Model 2 on Test Set"):
        output = model(batch, device)
        
        labels = ["None" if str(x).lower() == "nan" else str(x).strip() for x in batch["labels"]]
        predictions = output["predictions"][0]
        predicted_names = [id2label[x] for x in predictions]
        
        if len(labels) != len(predicted_names):
            if len(predicted_names) < len(labels):
                shortfall = len(labels) - len(predicted_names)
                predicted_names.extend([predicted_names[-1] if predicted_names else "NONE"] * shortfall)
            else:
                predicted_names = predicted_names[:len(labels)]
                
        true_labels.extend(labels)
        pred_labels.extend(predicted_names)

print("\n--- Final Model 2 Benchmarking Classification Report ---")
print(classification_report(true_labels, pred_labels, zero_division=0))


 LAUNCHING FINAL TRANS-MODEL BENCHMARK TEST EVALUATION
[Data] Fetching opennyaiorg/InRhetoricalRoles (test split)...
 -> Compiled 50 document sequences.


Evaluating Model 2 on Test Set:   0%|          | 0/50 [00:00<?, ?it/s]


--- Final Model 2 Benchmarking Classification Report ---
                precision    recall  f1-score   support

      ANALYSIS       0.63      0.90      0.74      1516
ARG_PETITIONER       0.55      0.68      0.61       324
ARG_RESPONDENT       0.00      0.00      0.00        95
           FAC       0.86      0.72      0.78      1261
         ISSUE       0.98      0.81      0.89        67
          NONE       0.88      0.76      0.82       399
      PREAMBLE       0.98      0.84      0.91      1290
PRE_NOT_RELIED       0.00      0.00      0.00         2
    PRE_RELIED       0.60      0.73      0.66       263
         RATIO       0.77      0.22      0.34       153
           RLC       0.69      0.43      0.53       143
           RPC       0.84      0.82      0.83       240
           STA       1.00      0.37      0.54        90

      accuracy                           0.76      5843
     macro avg       0.68      0.56      0.59      5843
  weighted avg       0.78      0.76      0.7